# Deep Q-Network (DQN) from Scratch: CartPole-v1

This notebook implements DQN completely from scratch using **PyTorch** and **Gymnasium**.
No RL libraries — every component is written and explained line by line.

**What you will build:**
1. CartPole environment walkthrough
2. Q-Network (neural net) in PyTorch
3. Experience Replay Buffer
4. Target Network (with periodic hard sync)
5. Full DQN training loop
6. Training curves and diagnostics
7. Evaluation: watch the trained agent balance the pole
8. Ablation: effect of removing replay buffer / target network

## Cell 1 — Imports

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')
print(f'Gymnasium version: {gym.__version__}')

## Cell 2 — CartPole Environment Walkthrough

Before building the agent, understand exactly what the environment provides.

**State vector (4 floats):**
- `state[0]` = cart position (metres)
- `state[1]` = cart velocity (m/s)
- `state[2]` = pole angle (radians, 0 = vertical)
- `state[3]` = pole angular velocity (rad/s)

**Actions:** `0` = push left, `1` = push right

**Reward:** +1 every step the pole stays up

**Termination:** pole angle > 12° OR cart position > 2.4m OR 500 steps

In [ ]:
env = gym.make('CartPole-v1')
env.reset(seed=SEED)

print('=== CartPole-v1 Environment ===')
print(f'Observation space : {env.observation_space}')
print(f'  Shape           : {env.observation_space.shape}')
print(f'  Low             : {env.observation_space.low}')
print(f'  High            : {env.observation_space.high}')
print()
print(f'Action space      : {env.action_space}')
print(f'  n_actions       : {env.action_space.n}')
print()

# Run one random episode to demonstrate structure
state, _ = env.reset(seed=SEED)
print(f'Initial state: {state}')
print(f'  cart_pos={state[0]:.4f}  cart_vel={state[1]:.4f}  '
      f'pole_angle={state[2]:.4f}  pole_vel={state[3]:.4f}')
print()

total_reward = 0
step = 0
for step in range(20):
    action = env.action_space.sample()   # random action
    next_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_reward += reward
    if step < 5:
        print(f'Step {step+1:2d}: action={action} ("{"LEFT" if action==0 else "RIGHT"}") '
              f'| reward={reward} | done={done}')
        print(f'         state={next_state.round(4)}')
    state = next_state
    if done:
        break

print(f'  ... episode lasted {step+1} steps, total reward = {total_reward}')
env.close()

## Cell 3 — Q-Network: The Neural Approximator

The Q-Network replaces the Q-table. It maps **state → Q-values for all actions**.

**Architecture:** Input(4) → Dense(128, ReLU) → Dense(128, ReLU) → Output(2)

Key design choices:
- **No output activation**: Q-values are unbounded real numbers, so we use no activation on the last layer
- **Single forward pass returns all action Q-values** simultaneously — efficient and required for computing `max_a' Q(s', a')`
- **ReLU** hidden activations: simple, effective for RL; avoids vanishing gradients

In [ ]:
class QNetwork(nn.Module):
    """
    Fully-connected Q-Network.
    
    Input  : state vector of shape (batch_size, n_states)
    Output : Q-values of shape (batch_size, n_actions)
    
    Q(s, a; θ) — one output per action, all computed simultaneously.
    """
    
    def __init__(self, n_states, n_actions, hidden_size=128):
        super(QNetwork, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(n_states, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_actions)
            # NO activation here — Q-values are unbounded real numbers
        )
        
        # Weight initialization: helps with early training stability
        self._init_weights()
    
    def forward(self, x):
        """
        x : tensor of shape (batch_size, n_states)
        Returns Q-values of shape (batch_size, n_actions)
        """
        return self.network(x)
    
    def _init_weights(self):
        for layer in self.network:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')
                nn.init.zeros_(layer.bias)


# ── Inspect the network ──────────────────────────────────────────────────────
n_states  = env.observation_space.shape[0]   # 4
n_actions = env.action_space.n               # 2

q_net = QNetwork(n_states, n_actions).to(device)
print('Q-Network Architecture:')
print(q_net)
print()

# Count parameters
total_params = sum(p.numel() for p in q_net.parameters())
print(f'Total trainable parameters: {total_params:,}')
print()

# Forward pass example
dummy_state = torch.tensor([[0.1, 0.0, -0.05, 0.0]], dtype=torch.float32).to(device)
q_values = q_net(dummy_state)
print(f'Example forward pass:')
print(f'  Input state  : {dummy_state.cpu().numpy()}')
print(f'  Output Q-vals: {q_values.detach().cpu().numpy()}')
print(f'  Best action  : {q_values.argmax().item()} ({"LEFT" if q_values.argmax().item()==0 else "RIGHT"})')
print()
print('(Q-values are random at initialization — this is expected)')

## Cell 4 — Experience Replay Buffer

The buffer stores past transitions `(s, a, r, s', done)` and allows **random mini-batch sampling**.

**Key design:**
- Fixed-capacity FIFO queue using `deque(maxlen=capacity)` — oldest entries auto-removed
- `sample(batch_size)` returns a batch of tensors ready for the network
- We return batch tensors directly (on the right device) for efficient training

**Why random sampling breaks correlation:**
Consecutive environment steps are highly correlated (same cart position, similar velocities).
Random sampling mixes transitions from many different time points, approximating i.i.d. data.

In [ ]:
class ReplayBuffer:
    """
    Fixed-capacity circular experience replay buffer.
    
    Stores (state, action, reward, next_state, done) tuples.
    Returns randomly-sampled mini-batches as PyTorch tensors.
    """
    
    def __init__(self, capacity=10_000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        """Store one transition."""
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        """
        Randomly sample batch_size transitions.
        Returns 5 tensors: states, actions, rewards, next_states, dones
        """
        transitions = random.sample(self.buffer, batch_size)
        
        # Unzip into separate lists
        states, actions, rewards, next_states, dones = zip(*transitions)
        
        # Convert to PyTorch tensors on the correct device
        states      = torch.tensor(np.array(states),      dtype=torch.float32).to(device)
        actions     = torch.tensor(np.array(actions),     dtype=torch.long   ).to(device)
        rewards     = torch.tensor(np.array(rewards),     dtype=torch.float32).to(device)
        next_states = torch.tensor(np.array(next_states), dtype=torch.float32).to(device)
        dones       = torch.tensor(np.array(dones),       dtype=torch.float32).to(device)
        
        return states, actions, rewards, next_states, dones
    
    def __len__(self):
        return len(self.buffer)
    
    def is_ready(self, batch_size):
        """True when we have enough samples to train."""
        return len(self) >= batch_size


# ── Test the buffer ──────────────────────────────────────────────────────────
buffer = ReplayBuffer(capacity=10_000)

# Push some dummy transitions
for _ in range(100):
    s  = np.random.rand(4).astype(np.float32)
    a  = np.random.randint(2)
    r  = 1.0
    s2 = np.random.rand(4).astype(np.float32)
    d  = False
    buffer.push(s, a, r, s2, d)

print(f'Buffer size after 100 pushes: {len(buffer)}')
states, actions, rewards, next_states, dones = buffer.sample(32)
print(f'Sampled batch shapes:')
print(f'  states      : {states.shape}')
print(f'  actions     : {actions.shape}')
print(f'  rewards     : {rewards.shape}')
print(f'  next_states : {next_states.shape}')
print(f'  dones       : {dones.shape}')

## Cell 5 — DQN Agent

The agent ties everything together:

1. **Two networks**: `online_net` (trained every step) and `target_net` (frozen, synced every C steps)
2. **`select_action`**: epsilon-greedy using the online network
3. **`compute_loss`**: MSE between Bellman target (from target_net) and prediction (from online_net)
4. **`train_step`**: backprop and optimizer step
5. **`sync_target_network`**: hard copy of online → target weights

In [ ]:
class DQNAgent:
    """
    Deep Q-Network agent.
    
    Components:
    - Online network Q(s,a; θ)       → trained via gradient descent
    - Target network Q(s,a; θ⁻)      → frozen, periodically synced from online
    - Experience replay buffer        → random mini-batch sampling
    - Epsilon-greedy action selection → exploration vs exploitation
    """
    
    def __init__(
        self,
        n_states,
        n_actions,
        lr=1e-3,              # learning rate for Adam optimizer
        gamma=0.99,           # discount factor
        epsilon=1.0,          # initial exploration rate
        epsilon_min=0.01,     # minimum exploration rate
        epsilon_decay=0.995,  # per-episode multiplicative decay
        buffer_size=10_000,   # replay buffer capacity
        batch_size=64,        # mini-batch size for training
        target_sync_freq=100  # sync target network every N steps
    ):
        self.n_states    = n_states
        self.n_actions   = n_actions
        self.gamma       = gamma
        self.epsilon     = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size  = batch_size
        self.target_sync_freq = target_sync_freq
        
        # ── Two networks ────────────────────────────────────────────────────
        self.online_net = QNetwork(n_states, n_actions).to(device)
        self.target_net = QNetwork(n_states, n_actions).to(device)
        
        # Target network starts as exact copy of online network
        self.sync_target_network()
        
        # Target network is never trained directly — freeze gradients
        for param in self.target_net.parameters():
            param.requires_grad = False
        
        # ── Optimizer ───────────────────────────────────────────────────────
        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)
        
        # ── Replay buffer ────────────────────────────────────────────────────
        self.buffer = ReplayBuffer(capacity=buffer_size)
        
        # ── Step counter (for target sync) ───────────────────────────────────
        self.step_count = 0
        
        # ── Diagnostic tracking ──────────────────────────────────────────────
        self.losses = []
    
    # ────────────────────────────────────────────────────────────────────────
    # Action selection: epsilon-greedy
    # ────────────────────────────────────────────────────────────────────────
    
    def select_action(self, state):
        """
        Epsilon-greedy action selection using the ONLINE network.
        
        Explore: with prob epsilon → random action
        Exploit: otherwise        → argmax Q(state, ·; θ)
        """
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        
        # Convert state to tensor, forward pass through online network
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = self.online_net(state_tensor)
        return q_values.argmax(dim=1).item()
    
    # ────────────────────────────────────────────────────────────────────────
    # Core training step
    # ────────────────────────────────────────────────────────────────────────
    
    def train_step(self):
        """
        Sample a mini-batch from the buffer and perform one gradient update.
        
        Loss = MSE(y_i, Q(s_i, a_i; θ))
        
        where y_i = r_i + gamma * max_a' Q(s'_i, a'; θ⁻)  [if not done]
              y_i = r_i                                      [if done]
        """
        if not self.buffer.is_ready(self.batch_size):
            return None   # Not enough data yet
        
        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)
        
        # ── Step 1: Compute Q(s, a; θ) for the actions taken ─────────────────
        # online_net outputs shape (batch, n_actions)
        # We want only the Q-value for the action that was actually taken
        # actions shape: (batch,) → unsqueeze → (batch, 1) for gather
        q_predicted = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        # q_predicted shape: (batch,)
        
        # ── Step 2: Compute Bellman targets using FROZEN target network ───────
        with torch.no_grad():
            # Q-values for all actions at next states, from TARGET network
            q_next = self.target_net(next_states)          # (batch, n_actions)
            max_q_next = q_next.max(dim=1).values          # (batch,) — best Q at s'
            
            # If done, future reward is 0 (episode ended, no future)
            # dones is 1.0 if done, 0.0 otherwise
            target = rewards + self.gamma * max_q_next * (1.0 - dones)
        # target shape: (batch,)
        
        # ── Step 3: Compute MSE loss ─────────────────────────────────────────
        loss = F.mse_loss(q_predicted, target)
        
        # ── Step 4: Backpropagate through ONLINE network only ─────────────────
        self.optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping: prevents exploding gradients (common DQN trick)
        nn.utils.clip_grad_norm_(self.online_net.parameters(), max_norm=10.0)
        
        self.optimizer.step()
        
        # ── Step 5: Periodically sync target network ──────────────────────────
        self.step_count += 1
        if self.step_count % self.target_sync_freq == 0:
            self.sync_target_network()
        
        loss_val = loss.item()
        self.losses.append(loss_val)
        return loss_val
    
    # ────────────────────────────────────────────────────────────────────────
    # Target network sync
    # ────────────────────────────────────────────────────────────────────────
    
    def sync_target_network(self):
        """
        Hard copy: θ⁻ ← θ
        The target network becomes an exact copy of the online network.
        """
        self.target_net.load_state_dict(self.online_net.state_dict())
    
    # ────────────────────────────────────────────────────────────────────────
    # Epsilon decay
    # ────────────────────────────────────────────────────────────────────────
    
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
    
    # ────────────────────────────────────────────────────────────────────────
    # Store transition
    # ────────────────────────────────────────────────────────────────────────
    
    def store(self, state, action, reward, next_state, done):
        self.buffer.push(state, action, reward, next_state, done)


# ── Quick sanity check ───────────────────────────────────────────────────────
agent = DQNAgent(n_states=n_states, n_actions=n_actions)
print('DQN Agent initialized.')
print(f'  Online network params : {sum(p.numel() for p in agent.online_net.parameters()):,}')
print(f'  Target network params : {sum(p.numel() for p in agent.target_net.parameters()):,}')
print(f'  Replay buffer capacity: {agent.buffer.buffer.maxlen:,}')
print(f'  Target sync every     : {agent.target_sync_freq} steps')
print()

# Verify online and target start identical
online_params = list(agent.online_net.parameters())[0].data
target_params = list(agent.target_net.parameters())[0].data
print(f'Online == Target at init: {torch.allclose(online_params, target_params)}')

## Cell 6 — Step-by-Step Trace: One Bellman Update

Before training, manually execute one complete DQN update to see every tensor and operation concretely.

In [ ]:
print('=== MANUAL DQN UPDATE TRACE ===')
print()

# Reset env and take one random step
env_trace = gym.make('CartPole-v1')
state, _ = env_trace.reset(seed=42)
action = 0   # LEFT
next_state, reward, terminated, truncated, _ = env_trace.step(action)
done = terminated or truncated

print(f'Transition:')
print(f'  state      = {state.round(4)}')
print(f'  action     = {action} (LEFT)')
print(f'  reward     = {reward}')
print(f'  next_state = {next_state.round(4)}')
print(f'  done       = {done}')
print()

# Build tensors (batch size 1 for illustration)
s_t  = torch.tensor(state,      dtype=torch.float32).unsqueeze(0).to(device)
s_t1 = torch.tensor(next_state, dtype=torch.float32).unsqueeze(0).to(device)
a_t  = torch.tensor([action],   dtype=torch.long   ).to(device)
r_t  = torch.tensor([reward],   dtype=torch.float32).to(device)
d_t  = torch.tensor([float(done)], dtype=torch.float32).to(device)

# Forward pass: online network
q_all      = agent.online_net(s_t)                               # (1, 2)
q_taken    = q_all.gather(1, a_t.unsqueeze(1)).squeeze(1)        # (1,) — Q(s, LEFT)

# Forward pass: target network (no grad)
with torch.no_grad():
    q_next_all = agent.target_net(s_t1)                          # (1, 2)
    max_q_next = q_next_all.max(dim=1).values                    # (1,)
    target     = r_t + agent.gamma * max_q_next * (1.0 - d_t)   # Bellman target

loss = F.mse_loss(q_taken, target)

print('--- Tensor values ---')
print(f'Online Q(s, all actions)   = {q_all.detach().cpu().numpy().round(4)}')
print(f'Online Q(s, LEFT taken)    = {q_taken.item():.6f}')
print()
print(f'Target Q(s\', all actions) = {q_next_all.cpu().numpy().round(4)}')
print(f'max Q(s\', *)              = {max_q_next.item():.6f}')
print()
print(f'Bellman target y = r + γ·max_Q = {reward} + {agent.gamma}×{max_q_next.item():.4f} = {target.item():.6f}')
print(f'TD error = y - Q(s,a)          = {target.item() - q_taken.item():.6f}')
print(f'MSE Loss = (y - Q(s,a))²       = {loss.item():.6f}')
print()
print('(Values are small random numbers from weight init — this is expected)')
env_trace.close()

## Cell 7 — Full Training Loop

**What happens each episode:**
1. Reset environment
2. For each step: select action (ε-greedy) → store transition → train step → check sync
3. After episode: decay ε, log metrics

**Convergence signal:** CartPole is considered "solved" when the agent achieves a mean reward of 475+ over 100 consecutive episodes.

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
HYPERPARAMS = dict(
    lr               = 1e-3,
    gamma            = 0.99,
    epsilon          = 1.0,
    epsilon_min      = 0.01,
    epsilon_decay    = 0.995,
    buffer_size      = 10_000,
    batch_size       = 64,
    target_sync_freq = 100,
)
N_EPISODES = 600
MAX_STEPS  = 500   # CartPole-v1 max is 500

# ── Initialize fresh agent and environment ────────────────────────────────────
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
env_train = gym.make('CartPole-v1')
agent = DQNAgent(n_states=n_states, n_actions=n_actions, **HYPERPARAMS)

# ── Metric tracking ───────────────────────────────────────────────────────────
ep_rewards    = []
ep_lengths    = []
epsilon_hist  = []
loss_hist     = []   # per-episode mean loss
sync_episodes = []   # episodes when target net was synced (for annotation)

print(f'Training DQN on CartPole-v1 for {N_EPISODES} episodes...')
print(f'Hyperparameters: {HYPERPARAMS}')
print()
print(f'{"Episode":>8}  {"Reward":>8}  {"Steps":>7}  {"Epsilon":>8}  '
      f'{"Buf":>6}  {"AvgR100":>9}  {"AvgLoss":>9}')
print('-' * 70)

for episode in range(N_EPISODES):
    state, _ = env_train.reset()
    ep_reward = 0
    ep_loss   = []
    prev_sync = agent.step_count // agent.target_sync_freq
    
    for step in range(MAX_STEPS):
        # 1. Select action
        action = agent.select_action(state)
        
        # 2. Step environment
        next_state, reward, terminated, truncated, _ = env_train.step(action)
        done = terminated or truncated
        
        # 3. Store transition in replay buffer
        agent.store(state, action, reward, next_state, float(done))
        
        # 4. Train (one gradient step)
        loss = agent.train_step()
        if loss is not None:
            ep_loss.append(loss)
        
        state      = next_state
        ep_reward += reward
        
        if done:
            break
    
    # 5. Decay epsilon
    agent.decay_epsilon()
    
    # Track metrics
    ep_rewards.append(ep_reward)
    ep_lengths.append(step + 1)
    epsilon_hist.append(agent.epsilon)
    loss_hist.append(np.mean(ep_loss) if ep_loss else 0.0)
    
    # Check if target was synced this episode
    if agent.step_count // agent.target_sync_freq > prev_sync:
        sync_episodes.append(episode)
    
    # Print every 50 episodes
    if (episode + 1) % 50 == 0:
        avg_r   = np.mean(ep_rewards[-100:])
        avg_l   = np.mean(loss_hist[-100:])
        print(f'{episode+1:>8d}  {ep_reward:>8.0f}  {step+1:>7d}  '
              f'{agent.epsilon:>8.3f}  {len(agent.buffer):>6d}  '
              f'{avg_r:>9.1f}  {avg_l:>9.4f}')

print()
print('Training complete!')
print(f'  Final epsilon       : {agent.epsilon:.4f}')
print(f'  Replay buffer size  : {len(agent.buffer):,}')
print(f'  Total gradient steps: {agent.step_count:,}')
print(f'  Target syncs        : {len(sync_episodes)}')
print(f'  Mean reward (last 100 eps): {np.mean(ep_rewards[-100:]):.1f}')

env_train.close()

## Cell 8 — Training Curves

Five plots:
1. Episode reward (+ rolling average)
2. Episode length (= steps before failure)
3. Rolling success rate (reward ≥ 195)
4. Epsilon decay
5. Training loss per episode

In [ ]:
def smooth(data, window=20):
    if len(data) < window:
        return np.array(data)
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('DQN Training on CartPole-v1', fontsize=15, fontweight='bold')
episodes = np.arange(N_EPISODES)

# 1. Episode reward
ax = axes[0, 0]
ax.plot(episodes, ep_rewards, alpha=0.25, color='steelblue', linewidth=0.8)
sm = smooth(ep_rewards, 20)
ax.plot(np.arange(len(sm)), sm, color='steelblue', linewidth=2.5, label='Smoothed (20-ep)')
ax.axhline(475, color='red', linestyle='--', alpha=0.6, label='Solved threshold (475)')
ax.set_title('Episode Reward'); ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# 2. Episode length
ax = axes[0, 1]
ax.plot(episodes, ep_lengths, alpha=0.25, color='coral', linewidth=0.8)
sm2 = smooth(ep_lengths, 20)
ax.plot(np.arange(len(sm2)), sm2, color='coral', linewidth=2.5, label='Smoothed (20-ep)')
ax.axhline(500, color='green', linestyle='--', alpha=0.6, label='Max (500 steps)')
ax.set_title('Episode Length (steps)'); ax.set_xlabel('Episode'); ax.set_ylabel('Steps')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# 3. Rolling success rate (reward >= 195 = "good episode")
ax = axes[1, 0]
success = [1 if r >= 195 else 0 for r in ep_rewards]
rolling = [np.mean(success[max(0, i-50):i+1]) * 100 for i in range(len(success))]
ax.plot(episodes, rolling, color='green', linewidth=2)
ax.axhline(90, color='green', linestyle='--', alpha=0.4, label='90%')
ax.fill_between(episodes, rolling, alpha=0.1, color='green')
ax.set_title('Rolling Success Rate (reward ≥ 195, 50-ep window)')
ax.set_xlabel('Episode'); ax.set_ylabel('Success Rate (%)')
ax.set_ylim(0, 105); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# 4. Epsilon decay
ax = axes[1, 1]
ax.plot(episodes, epsilon_hist, color='purple', linewidth=2)
ax.fill_between(episodes, epsilon_hist, alpha=0.15, color='purple')
ax.axhline(HYPERPARAMS['epsilon_min'], color='red', linestyle='--',
           label=f'ε_min={HYPERPARAMS["epsilon_min"]}')
ax.set_title('Epsilon Decay'); ax.set_xlabel('Episode'); ax.set_ylabel('Epsilon (ε)')
ax.set_ylim(0, 1.05); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# 5. Training loss
ax = axes[2, 0]
nonzero_loss = [(i, l) for i, l in enumerate(loss_hist) if l > 0]
if nonzero_loss:
    idxs, vals = zip(*nonzero_loss)
    ax.plot(idxs, vals, alpha=0.3, color='orange', linewidth=0.8)
    sm3 = smooth(vals, 20)
    ax.plot(np.array(idxs[:len(sm3)]), sm3, color='orange', linewidth=2.5, label='Smoothed (20-ep)')
ax.set_title('Mean Training Loss per Episode')
ax.set_xlabel('Episode'); ax.set_ylabel('MSE Loss')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# 6. Reward distribution (late vs early training)
ax = axes[2, 1]
split = N_EPISODES // 2
ax.hist(ep_rewards[:split],  bins=20, alpha=0.6, color='salmon',  label=f'First {split} eps')
ax.hist(ep_rewards[split:],  bins=20, alpha=0.6, color='steelblue', label=f'Last {split} eps')
ax.set_title('Reward Distribution: Early vs Late Training')
ax.set_xlabel('Episode Reward'); ax.set_ylabel('Count')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Cell 9 — Q-Value Diagnostics

Inspect what the trained network actually outputs.

We pass a fixed set of states through the trained network to see the Q-values and which action is preferred.

In [ ]:
print('=== Q-VALUE INSPECTION: TRAINED NETWORK ===')
print()

# Representative states for CartPole
test_states = {
    'Balanced (upright)    ' : np.array([0.0,  0.0,  0.0,  0.0], dtype=np.float32),
    'Tilting right (+angle)' : np.array([0.0,  0.0,  0.15, 0.3], dtype=np.float32),
    'Tilting left  (-angle)' : np.array([0.0,  0.0, -0.15,-0.3], dtype=np.float32),
    'Moving right fast     ' : np.array([0.5,  2.0,  0.05, 0.1], dtype=np.float32),
    'Moving left fast      ' : np.array([-0.5,-2.0, -0.05,-0.1], dtype=np.float32),
    'Near right wall       ' : np.array([2.0,  0.5,  0.0,  0.0], dtype=np.float32),
}

print(f'{"State Description":26}  {"Q(LEFT)":>10}  {"Q(RIGHT)":>10}  {"Best Action":>12}  {"Q Diff":>8}')
print('-' * 75)

for desc, state in test_states.items():
    s_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        q_vals = agent.online_net(s_t).cpu().numpy()[0]
    best  = 'LEFT' if q_vals[0] > q_vals[1] else 'RIGHT'
    diff  = abs(q_vals[0] - q_vals[1])
    print(f'{desc}  {q_vals[0]:>10.4f}  {q_vals[1]:>10.4f}  {best:>12}  {diff:>8.4f}')

print()
print('Interpretation:')
print('- "Tilting right (+angle)" → agent prefers RIGHT (corrective push)')
print('- "Tilting left (-angle)"  → agent prefers LEFT  (corrective push)')
print('- "Near right wall"        → agent prefers LEFT  (move away from boundary)')
print()
print('The Q-value DIFFERENCE (confidence) matters as much as the sign.')

# ── Also visualize Q-values as function of pole angle ────────────────────────
angles = np.linspace(-0.3, 0.3, 100)
q_left_vals  = []
q_right_vals = []

for angle in angles:
    state = np.array([0.0, 0.0, angle, 0.0], dtype=np.float32)
    s_t   = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        q = agent.online_net(s_t).cpu().numpy()[0]
    q_left_vals.append(q[0])
    q_right_vals.append(q[1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(np.degrees(angles), q_left_vals,  label='Q(s, LEFT)',  color='steelblue', linewidth=2)
ax1.plot(np.degrees(angles), q_right_vals, label='Q(s, RIGHT)', color='coral',     linewidth=2)
ax1.axvline(0, color='gray', linestyle='--', alpha=0.5, label='Upright (0°)')
ax1.set_xlabel('Pole Angle (degrees)')
ax1.set_ylabel('Q-value')
ax1.set_title('Q-Values vs Pole Angle\n(cart at center, zero velocity)')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.plot(np.degrees(angles),
         np.array(q_right_vals) - np.array(q_left_vals),
         color='purple', linewidth=2)
ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax2.axvline(0, color='gray', linestyle='--', alpha=0.5, label='Upright')
ax2.set_xlabel('Pole Angle (degrees)')
ax2.set_ylabel('Q(RIGHT) − Q(LEFT)')
ax2.set_title('Action Preference vs Pole Angle\n(positive = prefer RIGHT)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('Trained DQN Q-Values', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Cell 10 — Evaluate the Trained Agent

Run 20 evaluation episodes with pure greedy policy (ε = 0) and report performance.

In [ ]:
def evaluate_agent(agent, n_eval=20, verbose=True):
    """Run evaluation episodes with greedy policy (no exploration)."""
    env_eval = gym.make('CartPole-v1')
    saved_eps = agent.epsilon
    agent.epsilon = 0.0   # Pure greedy — no exploration
    
    rewards = []
    for ep in range(n_eval):
        state, _ = env_eval.reset()
        ep_reward = 0
        for step in range(500):
            action = agent.select_action(state)
            state, reward, terminated, truncated, _ = env_eval.step(action)
            ep_reward += reward
            if terminated or truncated:
                break
        rewards.append(ep_reward)
        if verbose and (ep < 5 or ep_reward < 400):
            print(f'  Eval episode {ep+1:2d}: reward = {ep_reward:.0f}  '
                  f'steps = {step+1}  {"✓" if ep_reward >= 475 else "✗"}')
    
    agent.epsilon = saved_eps  # Restore
    env_eval.close()
    return rewards


print('=== EVALUATION: TRAINED DQN (ε=0) ===')
print()
eval_rewards = evaluate_agent(agent, n_eval=20)

print()
print(f'Evaluation over 20 episodes:')
print(f'  Mean reward  : {np.mean(eval_rewards):.1f}')
print(f'  Std reward   : {np.std(eval_rewards):.1f}')
print(f'  Min reward   : {np.min(eval_rewards):.0f}')
print(f'  Max reward   : {np.max(eval_rewards):.0f}')
print(f'  Episodes ≥ 475: {sum(r >= 475 for r in eval_rewards)}/20')
print(f'  Episodes = 500: {sum(r == 500 for r in eval_rewards)}/20')
print()

# Bar chart of eval rewards
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['green' if r >= 475 else 'coral' for r in eval_rewards]
ax.bar(range(1, len(eval_rewards)+1), eval_rewards, color=colors, edgecolor='white')
ax.axhline(475, color='red', linestyle='--', linewidth=1.5, label='Solved (475)')
ax.axhline(np.mean(eval_rewards), color='blue', linestyle='-', linewidth=1.5,
           label=f'Mean ({np.mean(eval_rewards):.0f})')
ax.set_xlabel('Evaluation Episode'); ax.set_ylabel('Reward')
ax.set_title('Evaluation Rewards (Green = solved, Coral = partial)')
ax.set_xticks(range(1, len(eval_rewards)+1))
ax.set_ylim(0, 520); ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Cell 11 — Ablation Study: What Happens Without the Two Key DQN Components?

This is the most educational cell. We train three agents:
- **Full DQN** — both replay buffer and target network (baseline)
- **No Target Network** — online net used for both prediction and target computation
- **No Replay Buffer** — train on single most-recent transition (no random sampling)

⚠️ *This takes ~2-3 minutes. The instability you'll see in the ablated agents is exactly why these mechanisms were invented.*

In [ ]:
class DQNAgentAblation(DQNAgent):
    """DQN agent with optional disabling of replay buffer or target network."""
    
    def __init__(self, use_replay=True, use_target=True, **kwargs):
        super().__init__(**kwargs)
        self.use_replay = use_replay
        self.use_target = use_target
    
    def train_step_ablation(self, state=None, action=None, reward=None,
                            next_state=None, done=None):
        """
        Modified train step:
        - If use_replay=False: train on single provided transition (no buffer)
        - If use_target=False: use online net for target computation
        """
        if self.use_replay:
            if not self.buffer.is_ready(self.batch_size):
                return None
            states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)
        else:
            # No replay: train on single current transition
            if state is None:
                return None
            states      = torch.tensor([state],      dtype=torch.float32).to(device)
            actions     = torch.tensor([action],     dtype=torch.long   ).to(device)
            rewards     = torch.tensor([reward],     dtype=torch.float32).to(device)
            next_states = torch.tensor([next_state], dtype=torch.float32).to(device)
            dones       = torch.tensor([float(done)],dtype=torch.float32).to(device)
        
        q_predicted = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        
        with torch.no_grad():
            if self.use_target:
                q_next = self.target_net(next_states).max(dim=1).values
            else:
                # No target network: use online network (unstable!)
                q_next = self.online_net(next_states).max(dim=1).values
            target = rewards + self.gamma * q_next * (1.0 - dones)
        
        loss = F.mse_loss(q_predicted, target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.online_net.parameters(), max_norm=10.0)
        self.optimizer.step()
        
        self.step_count += 1
        if self.use_target and self.step_count % self.target_sync_freq == 0:
            self.sync_target_network()
        
        return loss.item()


def run_ablation(use_replay, use_target, n_episodes=400, label=''):
    """Train an ablated agent and return reward history."""
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    env_abl = gym.make('CartPole-v1')
    agent_abl = DQNAgentAblation(
        n_states=n_states, n_actions=n_actions,
        use_replay=use_replay, use_target=use_target,
        **HYPERPARAMS
    )
    rewards = []
    for ep in range(n_episodes):
        state, _ = env_abl.reset()
        ep_reward = 0
        for step in range(500):
            action = agent_abl.select_action(state)
            next_state, reward, terminated, truncated, _ = env_abl.step(action)
            done = terminated or truncated
            agent_abl.store(state, action, reward, next_state, float(done))
            # Pass current transition for no-replay case
            agent_abl.train_step_ablation(
                state=state, action=action, reward=reward,
                next_state=next_state, done=done
            )
            state = next_state; ep_reward += reward
            if done: break
        agent_abl.decay_epsilon()
        rewards.append(ep_reward)
    env_abl.close()
    print(f'  {label:35s} | Final mean (last 100): {np.mean(rewards[-100:]):6.1f}')
    return rewards


print('Running ablation study (3 agents × 400 episodes)...')
print()

r_full       = run_ablation(True,  True,  label='Full DQN (replay + target)')
r_no_target  = run_ablation(True,  False, label='No Target Network')
r_no_replay  = run_ablation(False, True,  label='No Replay Buffer')

print()

# Plot comparison
def rolling_mean(data, w=30):
    return [np.mean(data[max(0,i-w):i+1]) for i in range(len(data))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Ablation Study: Effect of DQN Components', fontsize=13, fontweight='bold')

eps = np.arange(400)
ax1.plot(eps, rolling_mean(r_full),       color='green',    linewidth=2.5, label='Full DQN')
ax1.plot(eps, rolling_mean(r_no_target),  color='red',      linewidth=2, linestyle='--', label='No Target Network')
ax1.plot(eps, rolling_mean(r_no_replay),  color='orange',   linewidth=2, linestyle='-.', label='No Replay Buffer')
ax1.axhline(475, color='gray', linestyle=':', alpha=0.7, label='Solved (475)')
ax1.set_title('Rolling Mean Reward (30-ep window)')
ax1.set_xlabel('Episode'); ax1.set_ylabel('Mean Reward')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# Final distribution comparison
data = [r_full[-100:], r_no_target[-100:], r_no_replay[-100:]]
labels = ['Full DQN', 'No Target\nNetwork', 'No Replay\nBuffer']
colors = ['green', 'red', 'orange']
bp = ax2.boxplot(data, labels=labels, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax2.axhline(475, color='gray', linestyle=':', alpha=0.7, label='Solved (475)')
ax2.set_title('Reward Distribution (Last 100 Episodes)')
ax2.set_ylabel('Episode Reward')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Cell 12 — Hyperparameter Sensitivity: Target Sync Frequency

One of the most impactful DQN hyperparameters: how often should we sync the target network?

- Too frequent (small C) → targets move too fast → instability
- Too infrequent (large C) → targets too stale → slow learning
- Sweet spot for CartPole: C ~ 50–200 steps

In [ ]:
def run_with_sync_freq(sync_freq, n_episodes=400):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    env_s = gym.make('CartPole-v1')
    agent_s = DQNAgent(n_states=n_states, n_actions=n_actions,
                       **{**HYPERPARAMS, 'target_sync_freq': sync_freq})
    rewards = []
    for ep in range(n_episodes):
        state, _ = env_s.reset()
        ep_reward = 0
        for _ in range(500):
            action = agent_s.select_action(state)
            ns, r, t, tr, _ = env_s.step(action)
            done = t or tr
            agent_s.store(state, action, r, ns, float(done))
            agent_s.train_step()
            state = ns; ep_reward += r
            if done: break
        agent_s.decay_epsilon()
        rewards.append(ep_reward)
    env_s.close()
    return rewards

sync_freqs = [10, 50, 100, 500]
print('Testing target sync frequencies:', sync_freqs)
results = {}
for sf in sync_freqs:
    results[sf] = run_with_sync_freq(sf)
    print(f'  sync_freq={sf:4d} | final mean: {np.mean(results[sf][-100:]):6.1f}')

fig, ax = plt.subplots(figsize=(12, 5))
colors_sync = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']
for sf, color in zip(sync_freqs, colors_sync):
    rm = rolling_mean(results[sf], 30)
    ax.plot(rm, color=color, linewidth=2, label=f'Target sync every {sf} steps')
ax.axhline(475, color='gray', linestyle=':', alpha=0.7)
ax.set_title('Effect of Target Network Sync Frequency on CartPole')
ax.set_xlabel('Episode'); ax.set_ylabel('Rolling Mean Reward (30-ep)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cell 13 — Summary and Next Steps

### What we built from scratch

| Component | Class/Function | Key Design Decision |
|-----------|---------------|---------------------|
| Neural Q-function | `QNetwork` | No output activation; outputs all Q(s,a) in one pass |
| Replay buffer | `ReplayBuffer` | Fixed deque; random mini-batch sampling; tensor conversion |
| DQN Agent | `DQNAgent.select_action` | ε-greedy using online network |
| Bellman update | `DQNAgent.train_step` | Uses target net for y, online net for prediction; stop-gradient on y |
| Target sync | `DQNAgent.sync_target_network` | Hard copy θ⁻ ← θ every C steps |
| Training loop | episode loop | Store → train → sync → decay ε |

### The two innovations, summarized

| Problem | Symptom | Solution |
|---------|---------|----------|
| Correlated transitions | Network forgets old states; oscillates | Experience Replay Buffer (random sampling) |
| Moving target | Training diverges or oscillates | Target Network (frozen for C steps) |

### Next steps to explore

- **Double DQN**: Use online net to *select* action, target net to *evaluate* it → reduces Q-value overestimation
- **Dueling DQN**: Separate the network into Value stream V(s) and Advantage stream A(s,a)
- **Prioritized Experience Replay**: Sample transitions with high TD error more frequently
- **Atari DQN**: Replace the MLP with a CNN; stack 4 frames as input
- **Policy Gradient (PPO)**: Directly optimize the policy instead of learning Q-values